# SolarVoix — Ingestion Bronze et enrichissement Silver

**Notebook Fabric :** `NB_Bronze_to_Silver`

**Lakehouse requis :** attachez `LH_SolarVoix` comme Lakehouse par défaut avant l'exécution.

Exécutez les cellules dans l'ordre. Pour la validation guidée, arrêtez-vous après la cellule 3 avant de commencer l'enrichissement IA.


## Bloc 3 — Ingestion Bronze


### Cellule 1 — Chargement Bronze des transcriptions


In [ ]:
# === CELLULE 1 : Ingestion Bronze — Fichiers texte ===
#
# OBJECTIF : Lire tous les fichiers .txt du dossier audio_calls/
#            en une seule opération Spark et créer la table bronze_transcriptions.
#
# spark.read.text() lit chaque fichier comme une ligne par ligne.
# Pour avoir une ligne par FICHIER (= un appel complet), on utilise
# spark.read.format("binaryFile") qui charge le contenu binaire du fichier,
# puis on le décode en UTF-8.
#
# Colonnes créées automatiquement par binaryFile :
#   path    → chemin complet du fichier (ex: abfss://...CALL_0001.txt)
#   content → contenu brut en bytes
#   length  → taille du fichier en bytes

from pyspark.sql import functions as F

# Lecture de tous les fichiers .txt en mode binaryFile
df_raw = (
    spark.read
    .format("binaryFile")
    .option("pathGlobFilter", "*.txt")  # filtre sur l'extension
    .load("Files/audio_calls/")
)

# Décodage UTF-8 + extraction du call_id depuis le nom du fichier
# F.decode(content, "utf-8") convertit les bytes en chaîne lisible
# F.regexp_extract extrait "CALL_0001" depuis le chemin complet du fichier
df_bronze_transcriptions = (
    df_raw
    .withColumn("transcription_text", F.decode(F.col("content"), "utf-8"))
    .withColumn("call_id",
        F.regexp_extract(F.col("path"), r"(CALL_\d+)\.txt", 1))
    .withColumn("file_size_bytes", F.col("length"))
    .select("call_id", "transcription_text", "file_size_bytes")
)

df_bronze_transcriptions.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("bronze_transcriptions")

print(f"bronze_transcriptions : {df_bronze_transcriptions.count()} lignes")
df_bronze_transcriptions.show(3, truncate=80)


### Cellule 2 — Chargement Bronze des métadonnées


In [ ]:
# === CELLULE 2 : Ingestion Bronze — Métadonnées CSV ===
#
# OBJECTIF : Lire le fichier calls_metadata.csv et créer bronze_calls_metadata.
#
# inferSchema=True détecte automatiquement :
#   - date_appel → TimestampType (si format reconnu)
#   - duree_secondes → IntegerType
#   - les colonnes texte → StringType
#
# Bonne pratique : en production, définir un StructType explicite
# pour éviter les surprises de typage sur de gros volumes.

df_meta = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/metadata/calls_metadata.csv")
)

df_meta.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("bronze_calls_metadata")

print(f"bronze_calls_metadata : {df_meta.count()} lignes")
df_meta.printSchema()


### Cellule 3 — Diagnostic qualité Bronze


In [ ]:
# === CELLULE 3 : Diagnostic qualité ===
#
# OBJECTIF : Vérifier l'intégrité des données avant de commencer l'enrichissement IA.
#
# Règle d'or : diagnostiquer avant de transformer.
# On vérifie :
#   1. Nulls dans les colonnes critiques
#   2. Cohérence du nombre d'appels (120 attendus)
#   3. Jointure entre les deux tables Bronze (call_id doit matcher)

from pyspark.sql import functions as F

df_t = spark.table("bronze_transcriptions")
df_m = spark.table("bronze_calls_metadata")

print("=== VÉRIFICATION BRONZE ===")
print(f"Transcriptions : {df_t.count()} lignes (attendu: 120)")
print(f"Métadonnées    : {df_m.count()} lignes (attendu: 120)")

# Vérification des nulls sur transcription_text (critique pour l'IA)
nulls_text = df_t.filter(F.col("transcription_text").isNull() |
                          (F.length("transcription_text") < 10)).count()
print(f"\nTranscriptions vides ou trop courtes : {nulls_text}")

# Vérification de la jointure (call_ids qui matchent)
ids_t = set(r.call_id for r in df_t.select("call_id").collect())
ids_m = set(r.call_id for r in df_m.select("call_id").collect())
orphelins = ids_t.symmetric_difference(ids_m)
print(f"call_ids sans correspondance : {len(orphelins)}")
if len(orphelins) > 0:
    print(f"  ⚠️ IDs orphelins : {list(orphelins)[:5]}")
else:
    print("  ✅ Jointure parfaite entre les deux tables")

# Distribution des sentiments réels (vérité terrain)
print("\n=== RÉPARTITION SENTIMENTS (vérité terrain) ===")
df_m.groupBy("sentiment_reel").count().orderBy("count", ascending=False).show()


## Bloc 4 — Enrichissement IA


### Cellule 4A — Installer le moteur IA et redémarrer Python

> ⚠️ Exécutez cette installation dans une cellule séparée. Le redémarrage empêche `transformers` de conserver un état où `torch` était absent.


In [ ]:
# === CELLULE 4A : Installation des dépendances IA ===
%pip install -q "transformers[torch]"

# Redémarrage obligatoire après l'installation.
# Le contexte Spark et les tables Bronze restent disponibles.
notebookutils.session.restartPython()


### Cellule 4B — Analyse de sentiment IA

Attendre la fin du redémarrage Python, puis exécuter cette cellule.


In [ ]:
# === CELLULE 4B : Analyse de sentiment avec HuggingFace Transformers ===
#
# OBJECTIF : Attribuer un score de sentiment (positif/neutre/négatif)
#            à chaque transcription via un modèle de NLP pré-entraîné.
#
# Modèle : cardiffnlp/twitter-xlm-roberta-base-sentiment
#   → Entraîné sur 198 millions de tweets en 100 langues dont le français
#   → Labels : negative / neutral / positive
#   → Taille : ~280 MB (téléchargé automatiquement au premier appel)
#
# Stratégie d'exécution :
#   - On collecte les transcriptions dans le driver (120 textes courts = OK)
#   - On applique le pipeline NLP en batch sur le driver
#   - Pour des millions de documents, on utiliserait une UDF Spark distribuée
#
# truncation=True, max_length=512 → les transcriptions longues sont tronquées
# à 512 tokens (limite du modèle BERT/RoBERTa)

import torch
from transformers import pipeline as hf_pipeline

print(f"PyTorch {torch.__version__} chargé ✅")

print("Chargement du modèle de sentiment (première fois : ~1 min)...")

# Modèle : nlptown/bert-base-multilingual-uncased-sentiment
# Tokenizer BERT standard (WordPiece) — aucune dépendance sentencepiece
# Supporte le français nativement (entraîné sur 6 langues)
# Labels de sortie : "1 star" à "5 stars"
sentiment_pipeline = hf_pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    device=-1,
    truncation=True,
    max_length=512
)

print("Modèle chargé ✅")

rows = spark.table("bronze_transcriptions").collect()
texts = [r.transcription_text for r in rows]
call_ids = [r.call_id for r in rows]

resultats_sentiment = sentiment_pipeline(texts, batch_size=16)

# Mapping : 1-2 étoiles → negatif | 3 étoiles → neutre | 4-5 étoiles → positif
def mapper_sentiment(label: str) -> str:
    nb = int(label[0])
    if nb <= 2:
        return "negatif"
    elif nb == 3:
        return "neutre"
    else:
        return "positif"

resultats = []
for call_id, texte, res in zip(call_ids, texts, resultats_sentiment):
    sentiment_ia = mapper_sentiment(res["label"])
    score_confiance = round(res["score"], 4)
    resultats.append({
        "call_id": call_id,
        "sentiment_ia": sentiment_ia,
        "score_sentiment": score_confiance,
    })

print(f"\nAnalyse terminée : {len(resultats)} appels traités")
# Aperçu des 5 premiers résultats
for r in resultats[:5]:
    print(f"  {r['call_id']} → {r['sentiment_ia']} (confiance: {r['score_sentiment']})")


### Cellule 5 — Détection d'intention et score de risque churn


In [ ]:
# === CELLULE 5 : Détection d'intention par règles + calcul du score churn ===
#
# OBJECTIF : Deux enrichissements complémentaires :
#   1. intent_detecte : catégorie d'intention basée sur mots-clés métier
#      (plus rapide et explicable qu'un modèle NLP pour la classification métier)
#   2. score_risque_churn : score de 0 à 100 combinant plusieurs signaux
#
# Logique du score risque churn :
#   - Sentiment IA très négatif  : +50 points
#   - Sentiment IA négatif       : +25 points
#   - Mots de résiliation        : +30 points
#   - Mots de menace (avocat, DGCCRF) : +20 points
#   - Durée > 5 minutes (frustration prolongée) : +10 points
#   - Score plafonné à 100
#
# Cette formule est délibérément simple et explicable.
# En production, on l'enrichirait avec l'historique CRM du client.

import re

# Dictionnaire des mots-clés par intention métier
KEYWORDS_INTENT = {
    "resiliation":         ["résilie", "résiliation", "annuler", "annule", "partir", "quitter"],
    "menace_juridique":    ["avocat", "plainte", "dgccrf", "médiateur", "tribunal", "juridique"],
    "churn_imminent":      ["jamais", "arnaque", "honte", "incompétent", "remboursement total"],
    "plainte_technique":   ["panne", "ne fonctionne pas", "cassé", "défaillant", "réparer"],
    "plainte_sav":         ["rappelle pas", "personne ne", "inacceptable", "scandaleux"],
    "question_facturation":["facture", "tva", "remboursement", "devis", "tarif", "prix"],
    "suivi_commande":      ["livraison", "délai", "date", "quand", "attends"],
    "satisfaction":        ["merci", "excellent", "parfait", "satisfait", "recommande"],
    "information":         ["information", "renseignement", "savoir", "comprendre"],
}

def detecter_intention(texte: str) -> str:
    """Retourne la première intention trouvée par ordre de priorité."""
    texte_lower = texte.lower()
    # Ordre de priorité : les intentions critiques d'abord
    priorite = ["resiliation", "menace_juridique", "churn_imminent",
                "plainte_technique", "plainte_sav", "question_facturation",
                "suivi_commande", "satisfaction", "information"]
    for intent in priorite:
        for kw in KEYWORDS_INTENT[intent]:
            if kw in texte_lower:
                return intent
    return "autre"

def calculer_score_churn(sentiment: str, texte: str, duree_sec: int) -> int:
    """
    Calcule un score de risque churn entre 0 et 100.
    Plus le score est élevé, plus le risque que le client parte est important.
    """
    score = 0
    texte_lower = texte.lower()

    # Signal 1 : Sentiment IA (base élevée — le sentiment est le signal le plus fort)
    if sentiment == "negatif":
        score += 40

    # Signal 2 : Mots de résiliation explicites
    mots_resil = ["résilie", "résiliation", "annuler", "quitter", "partir", "dernier"]
    if any(m in texte_lower for m in mots_resil):
        score += 30

    # Signal 3 : Menaces juridiques ou institutionnelles
    mots_menace = ["avocat", "plainte", "dgccrf", "médiateur", "tribunal", "assurance"]
    if any(m in texte_lower for m in mots_menace):
        score += 20

    # Signal 4 : Plainte technique ou SAV explicite
    mots_plainte = ["panne", "ne fonctionne pas", "inacceptable", "scandaleux",
                    "inadmissible", "trois fois", "cinq fois", "rappelle pas"]
    if any(m in texte_lower for m in mots_plainte):
        score += 15

    # Signal 5 : Durée prolongée (>5 min = client insistant = problème sérieux)
    if duree_sec > 300:
        score += 10

    # Plafonnement à 100
    return min(score, 100)

# -------------------------------------------------------
# APPLICATION SUR LES 120 APPELS
# -------------------------------------------------------
# On joint les résultats de sentiment avec les métadonnées (durée)
df_meta_pd = spark.table("bronze_calls_metadata").toPandas()
df_meta_dict = {r["call_id"]: r for _, r in df_meta_pd.iterrows()}

for r in resultats:
    texte = next(t for c, t in zip(call_ids, texts) if c == r["call_id"])
    meta = df_meta_dict.get(r["call_id"], {})
    duree = meta.get("duree_secondes", 0)

    r["intent_detecte"] = detecter_intention(texte)
    r["score_risque_churn"] = calculer_score_churn(r["sentiment_ia"], texte, duree)
    r["alerte_critique"] = r["score_risque_churn"] >= 50  # booléen pour Activator (Partie 2)

# Statistiques rapides
scores = [r["score_risque_churn"] for r in resultats]
alertes = sum(r["alerte_critique"] for r in resultats)
print(f"Score churn moyen : {sum(scores)/len(scores):.1f}/100")
print(f"Score churn max   : {max(scores)}/100")
print(f"Appels en alerte critique (score ≥ 50) : {alertes}")


## Bloc 5 — Couche Silver


### Cellule 6 — Construction de la table Silver


In [ ]:
# === CELLULE 6 : Construction de la table Silver ===
#
# OBJECTIF : Créer silver_appels_enrichis en joignant :
#   - bronze_calls_metadata  (informations client, produit, date)
#   - Les résultats d'enrichissement IA (sentiment, intent, churn)
#
# La couche Silver = données propres + enrichies, prêtes pour le rapport.
# On écrit en Delta → partitionnement par sentiment_ia pour accélérer
# les requêtes filtrées dans Power BI.
#
# Colonnes finales de la table Silver :
#   call_id, client_nom, client_email, ville, produit
#   date_appel, duree_secondes, duree_minutes (calculée)
#   sentiment_ia, score_sentiment
#   intent_detecte, score_risque_churn, alerte_critique
#   (sentiment_reel, intention_reelle → colonnes de vérité terrain pour audit)

from pyspark.sql import Row
import pyspark.sql.functions as F

# Conversion de la liste Python → DataFrame Spark
df_enrichi = spark.createDataFrame([Row(**r) for r in resultats])

# Jointure avec les métadonnées Bronze
df_meta_spark = spark.table("bronze_calls_metadata")

df_silver = (
    df_meta_spark
    .join(df_enrichi, on="call_id", how="inner")

    # Calcul de la durée en minutes (arrondi 1 décimale) pour lisibilité métier
    .withColumn("duree_minutes",
        F.round(F.col("duree_secondes") / 60.0, 1))

    # Conversion de la date en type Timestamp pour Power BI
    .withColumn("date_appel", F.to_timestamp("date_appel"))

    # Extraction jour de la semaine (1=Lundi ... 7=Dimanche) pour analyses
    .withColumn("jour_semaine", F.dayofweek("date_appel"))

    # Extraction heure pour analyser les pics d'appels
    .withColumn("heure_appel", F.hour("date_appel"))

    # Tranche horaire métier (matin / après-midi / fin de journée)
    .withColumn("tranche_horaire",
        F.when(F.col("heure_appel") < 12, "Matin")
        .when(F.col("heure_appel") < 17, "Après-midi")
        .otherwise("Fin de journée"))

    # Sélection et ordre des colonnes finales
    .select(
        "call_id", "client_nom", "client_email", "ville", "produit",
        "date_appel", "duree_secondes", "duree_minutes",
        "tranche_horaire", "jour_semaine",
        "sentiment_ia", "score_sentiment",
        "intent_detecte", "score_risque_churn", "alerte_critique",
        "sentiment_reel", "intention_reelle"  # colonnes audit
    )
)

# Écriture Silver partitionnée par sentiment_ia
# Avantage : Power BI en mode Direct Lake lit uniquement les partitions filtrées
df_silver.write \
    .mode("overwrite") \
    .format("delta") \
    .partitionBy("sentiment_ia") \
    .saveAsTable("silver_appels_enrichis")

print(f"silver_appels_enrichis : {df_silver.count()} lignes")
print("\nDistribution sentiment IA :")
df_silver.groupBy("sentiment_ia").count().orderBy("count", ascending=False).show()


### Cellule 7 — Contrôle qualité Silver


In [ ]:
# === CELLULE 7 : Contrôle qualité Silver ===
#
# OBJECTIF : Valider la qualité de la table Silver avant de passer à la visualisation.
# On vérifie les distributions, les scores extrêmes, et la cohérence métier.

df_s = spark.table("silver_appels_enrichis")

print("=== CONTRÔLE QUALITÉ SILVER ===\n")

# 1. Nulls sur les colonnes critiques
cols_critiques = ["sentiment_ia", "intent_detecte", "score_risque_churn"]
for col in cols_critiques:
    nulls = df_s.filter(F.col(col).isNull()).count()
    statut = "✅" if nulls == 0 else "⚠️"
    print(f"{statut} {col} : {nulls} nulls")

# 2. Distribution des scores churn
print("\n--- Distribution score_risque_churn ---")
df_s.select(
    F.min("score_risque_churn").alias("min"),
    F.max("score_risque_churn").alias("max"),
    F.avg("score_risque_churn").alias("moyenne"),
    F.percentile_approx("score_risque_churn", 0.75).alias("p75"),
    F.percentile_approx("score_risque_churn", 0.90).alias("p90"),
).show()

# 3. Appels en alerte critique par produit
print("--- Alertes critiques par produit ---")
df_s.filter(F.col("alerte_critique") == True) \
    .groupBy("produit") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

# 4. Cohérence : sentiment IA vs sentiment réel (matrice de confusion simplifiée)
print("--- Cohérence IA vs vérité terrain ---")
df_s.groupBy("sentiment_reel", "sentiment_ia").count() \
    .orderBy("sentiment_reel", "sentiment_ia") \
    .show()


## Bloc 7 — Optimisation


### Cellule 8 — Optimisation Delta


In [ ]:
# === CELLULE 8 : Optimisation Delta ===
#
# OBJECTIF : Optimiser la table Silver pour les lectures Power BI Direct Lake.
#
# OPTIMIZE : compacte les petits fichiers Parquet en fichiers de ~128-256 MB
#   → réduit le nombre de fichiers à lire pour Power BI
#   → améliore les temps de requête
#
# VACUUM (7 jours) : supprime les anciens fichiers Delta non référencés
#   → libère l'espace de stockage OneLake
#   ⚠️ Ne pas descendre sous 7 jours si vous avez du Time Travel actif

spark.sql("OPTIMIZE silver_appels_enrichis")
print("OPTIMIZE terminé")

spark.sql("VACUUM silver_appels_enrichis RETAIN 168 HOURS")
print("VACUUM terminé")

# Statistiques de la table après optimisation
spark.sql("DESCRIBE DETAIL silver_appels_enrichis") \
    .select("numFiles", "sizeInBytes", "partitionColumns") \
    .show(truncate=False)
